# Эксперимент: TF-IDF/LogReg vs GigaChat

Ноутбук использует существующую модель, authoritative taxonomy_v2, backend adapter и GigaChat-клиент проекта. Он не изменяет БД и по умолчанию не делает сетевых запросов. Цель — проверить, улучшает ли GigaChat качество и покрытие на независимой ручной разметке.

Запуск: откройте notebook в VS Code из репозитория и выберите Python kernel проекта. При необходимости: `pip install -r requirements-notebook.txt`.


## GigaChat API Configuration

1. Создайте проект GigaChat API в кабинете разработчика Сбера и получите OAuth **Authorization Key**.
2. Храните его только в локальном `.env` (файл исключён из Git): `RC_GIGACHAT_CREDENTIALS=...`, при необходимости задайте scope и URL как в `.env.example`.
3. Модель задаётся через `RC_GIGACHAT_MODEL`. Сначала выполните connection test: он читает список доступных моделей. Наличие `GigaChat-2` не предполагается.
4. Реальные запросы требуют одновременно двух opt-in флагов. Credentials/token не печатаются и не экспортируются.

Без credentials выполняются все локальные разделы и читаются ранее закешированные ответы.


In [ ]:
from __future__ import annotations
import asyncio, json, os, sys, time
from pathlib import Path
import matplotlib.pyplot as plt, numpy as np, pandas as pd
from IPython.display import display
from sklearn.metrics import ConfusionMatrixDisplay
from sqlalchemy import select

def find_root(start=None):
    p=(start or Path.cwd()).resolve()
    for q in (p,*p.parents):
        if (q/"app").is_dir() and (q/"ml_workspace").is_dir(): return q
    raise RuntimeError("Откройте notebook внутри репозитория IYM")
ROOT=find_root(); sys.path.insert(0,str(ROOT)) if str(ROOT) not in sys.path else None; os.chdir(ROOT)

from app.config import Settings
from app.db import get_sessionmaker
from app.llm.gigachat import GigaChatClient, GigaChatError
from app.ml.hybrid import decide_hybrid, eligible_for_gigachat
from app.ml.tfidf import TfidfClassifier, prediction_to_suggestion, product_to_input, publishable
from app.models import Product
from app.taxonomy import ProductCategory
from ml_workspace.ml_api import predict

OUT=ROOT/"ml_workspace"/"outputs"/"gigachat_matching_experiment"; OUT.mkdir(parents=True,exist_ok=True)
PRED_CSV=OUT/"catalog_predictions.csv"; SAMPLE_CSV=OUT/"representative_sample_240.csv"
TEMPLATE_CSV=OUT/"manual_annotation_template.csv"; HUMAN_CSV=OUT/"manual_annotations_completed.csv"
GIGA_CSV=OUT/"gigachat_results.csv"; USAGE_CSV=OUT/"gigachat_api_usage.csv"

# Network is off by default. A pilot requires both flags to be deliberately changed locally.
GIGACHAT_ENABLED=True
ALLOW_REAL_API_REQUESTS=True
GIGACHAT_MODEL=os.getenv("RC_GIGACHAT_MODEL","GigaChat-2")
MAX_API_PRODUCTS=30; MAX_TOKEN_BUDGET=50_000; USER_TOKEN_QUOTA=1_000_000
RANDOM_SEED=20261028; ML_THRESHOLD=.7; LOAD_WEAK_LABELS=True; RUN_OPTIONAL_EXPLANATION=False
print(ROOT,OUT)
display({"GIGACHAT_ENABLED":GIGACHAT_ENABLED,"ALLOW_REAL_API_REQUESTS":ALLOW_REAL_API_REQUESTS,
         "model":GIGACHAT_MODEL,"MAX_API_PRODUCTS":MAX_API_PRODUCTS,"MAX_TOKEN_BUDGET":MAX_TOKEN_BUDGET})


### Connection test

Ячейка аутентифицируется и читает `/models` только при двойном opt-in. Секреты не выводятся. Выберите реально доступный model ID из результата.


In [ ]:
import inspect

print(inspect.getfile(GigaChatClient))
print([name for name in dir(GigaChatClient) if not name.startswith("_")])


In [ ]:
# Refresh only when this kernel still holds a pre-discovery client class.
# The supported public connectivity interface is GigaChatClient.models().
import importlib
import app.llm.gigachat as gigachat_module
if not hasattr(GigaChatClient, "models"):
    importlib.reload(gigachat_module)
    GigaChatClient=gigachat_module.GigaChatClient
    GigaChatError=gigachat_module.GigaChatError

ca_env=os.getenv("RC_GIGACHAT_CA_BUNDLE")
settings=Settings(gigachat_enabled=GIGACHAT_ENABLED,gigachat_model=GIGACHAT_MODEL,
                  gigachat_token_budget=MAX_TOKEN_BUDGET,
                  gigachat_ca_bundle=Path(ca_env) if ca_env else None)
giga=GigaChatClient(settings)

async def connection_test():
    if not (GIGACHAT_ENABLED and ALLOW_REAL_API_REQUESTS):
        return {"status":"disabled","message":"Включите оба opt-in флага локально"}
    try:
        models=await giga.models()
        return {"status":"ok","selected":GIGACHAT_MODEL,
                "selected_available":GIGACHAT_MODEL in models,"available_models":models}
    except Exception as e:
        return {"status":"error","error":f"{type(e).__name__}: {e}"}
display(await connection_test())


## 1. Полный каталог и локальная ML-классификация

Используется канонический artifact и текущий backend mapping. Только `SELECT`: notebook не вызывает migration, commit или сохранение bindings. Локальный CSV ускоряет повторный запуск.


In [ ]:
ARTIFACT=ROOT/"ml_workspace"/"model"/"tfidf_logreg"/"model"
assert ARTIFACT.is_dir() and (ROOT/"repaircalc_local.sqlite").is_file()
def classify_catalog(batch=2000):
    clf=TfidfClassifier(ARTIFACT); rows=[]
    with get_sessionmaker()() as s:
        ids=list(s.scalars(select(Product.id).where(Product.status.isnot(False)).order_by(Product.id)))
        for start in range(0,len(ids),batch):
            ps=list(s.scalars(select(Product).where(Product.id.in_(ids[start:start+batch])).order_by(Product.id)))
            raw=predict(clf.model,[product_to_input(p) for p in ps])
            for p,r in zip(ps,raw,strict=True):
                x=prediction_to_suggestion(r,p); cs=r.dimension_scores.get("product_category",{})
                cat=max(cs,key=cs.get) if cs else ""
                rows.append({"product_id":p.id,"name":p.name,
                    "catalog_category":" | ".join((p.categories or {}).values()),
                    "description":(p.description or "")[:1500],"ml_domain":r.domain_label,"ml_status":r.status,
                    "ml_product_category":cat,
                    "backend_product_category":x.product_category.value if x.product_category else "",
                    "backend_material":x.material.value,"ml_confidence":x.confidence,
                    "ml_work_types":json.dumps([w.value for w in x.work_types],ensure_ascii=False),
                    "taxonomy_conflict":bool(x.flags.get("taxonomy_inconsistent")),
                    "unknown_domain":r.status in {"unknown","uncertain"},
                    "non_renovation":r.status=="non_renovation","furnishing":r.domain_label=="furnishing",
                    **{f"eligible_{str(t).replace('.','_')}":publishable(x,t) for t in (.6,.7,.8)}})
    return pd.DataFrame(rows)
if PRED_CSV.exists():
    pred=pd.read_csv(PRED_CSV,keep_default_na=False); source="cache"
else:
    pred=classify_catalog(); pred.to_csv(PRED_CSV,index=False); source="computed"
assert len(pred)==88790 and pred.product_id.is_unique
print(source,pred.shape); display(pred.head(3))


### Weak labels ≠ ground truth

Weak labels показываются отдельно и не используются как истинные ответы для accuracy.


In [ ]:
weak_path=ROOT/"ml_workspace"/"data"/"products_taxonomy_v3.csv"
if LOAD_WEAK_LABELS and weak_path.exists():
    weak=pd.read_csv(weak_path,usecols=["product_id","domain_label","product_category","work_types"],
                     low_memory=False).rename(columns={"domain_label":"weak_domain",
                     "product_category":"weak_product_category","work_types":"weak_work_types"})
    pred=pred.merge(weak,on="product_id",how="left",validate="one_to_one")
else:
    for c in ["weak_domain","weak_product_category","weak_work_types"]: pred[c]=""
print("weak labels loaded:",LOAD_WEAK_LABELS and weak_path.exists())


## 2. Репрезентативная выборка (240 товаров)

Фиксированный seed, непересекающиеся страты: ambiguous cases, conflicts, unknown, non-renovation, low/medium/high confidence. В sample сохраняются реальные ID, названия, категории, labels, scores и work types.


In [ ]:
pattern=r"затир|плиточн.*клей|профил|гипсокарт|рамк.*(розет|выключ)|розет|клей|краск|ламп|светиль|труб|фитинг"
amb={"tile","drywall","electrical_goods","adhesive","paint","lighting","plumbing"}
pool=pred.copy()
pool["is_ambiguous"]=pool.ml_product_category.isin(amb)&pool.name.str.contains(pattern,case=False,regex=True,na=False)
spec=[("ambiguous_known_cases",pool.is_ambiguous,50),("taxonomy_conflict",pool.taxonomy_conflict,30),
      ("unknown_or_uncertain",pool.unknown_domain,30),("non_renovation",pool.non_renovation,30),
      ("low_confidence",(pool.ml_confidence<.7)&~pool.non_renovation,40),
      ("medium_confidence",pool.ml_confidence.between(.7,.8,inclusive="left"),30),
      ("high_confidence",pool.ml_confidence>=.8,30)]
rng=np.random.default_rng(RANDOM_SEED); used=set(); parts=[]
for label,mask,n in spec:
    candidates=pool[mask&~pool.product_id.isin(used)]
    chosen=candidates.iloc[rng.choice(len(candidates),min(n,len(candidates)),replace=False)].copy()
    chosen["sampling_stratum"]=label; used.update(chosen.product_id.astype(int)); parts.append(chosen)
sample=pd.concat(parts,ignore_index=True).sort_values(["sampling_stratum","product_id"])
assert len(sample)==240 and sample.product_id.is_unique
sample.to_csv(SAMPLE_CSV,index=False)
display(sample.groupby("sampling_stratum").size().to_frame("products"))
display(sample[["product_id","name","catalog_category","ml_product_category","backend_material",
                "ml_confidence","ml_work_types","sampling_stratum"]].head(15))


## 3. Dataset overview и ML coverage

Coverage — наличие публикуемого решения, не accuracy. Score 0.7 является operating threshold, а не вероятностью правильности.


In [ ]:
display(pd.DataFrame({"metric":["catalog","sample","non_renovation","unknown","conflicts"],
"value":[len(pred),len(sample),pred.non_renovation.sum(),pred.unknown_domain.sum(),pred.taxonomy_conflict.sum()]}))
fig,ax=plt.subplots(1,2,figsize=(13,4))
pred.ml_confidence.plot.hist(bins=30,ax=ax[0],title="ML confidence distribution")
for t,c in [(.6,"green"),(.7,"orange"),(.8,"red")]: ax[0].axvline(t,color=c,ls="--")
coverage=pd.Series({t:pred[f"eligible_{str(t).replace('.','_')}"].mean() for t in (.6,.7,.8)})
coverage.plot.bar(ax=ax[1],ylim=(0,1),title="Automatic binding coverage"); plt.tight_layout(); plt.show()
cat_cov=pred.groupby("ml_product_category").agg(products=("product_id","size"),
    eligible=("eligible_0_6","sum"),mean_confidence=("ml_confidence","mean"))
cat_cov["coverage"]=cat_cov.eligible/cat_cov.products
display(cat_cov.sort_values("products",ascending=False))


## 4. GigaChat pilot: structured JSON, cache, retries and complete accounting

Default is offline. The same eligibility and validation policy is used in production and here.


In [ ]:
# Same eligibility rule as production; non-renovation/furnishing rows are not blanket-sent.
api_candidates=sample[(sample.taxonomy_conflict|sample.unknown_domain|(sample.ml_confidence<ML_THRESHOLD))
                      & ~sample.non_renovation & ~sample.furnishing].head(MAX_API_PRODUCTS)
usage_cols=["product_id","model","input_tokens","output_tokens","total_tokens","response_ms","status",
            "error_kind","error","from_cache","attempt"]
async def run_pilot(frame):
    old=pd.read_csv(GIGA_CSV,keep_default_na=False) if GIGA_CSV.exists() else pd.DataFrame()
    done=set(old.product_id.astype(int)) if len(old) else set(); results=old.to_dict("records")
    with get_sessionmaker()() as s:
        for row in frame.itertuples(index=False):
            if int(row.product_id) in done: continue
            p=s.get(Product,int(row.product_id))
            try:
                x=await giga.classify(p); error=""; error_kind=""
                results.append({"product_id":row.product_id,"giga_decision":x.decision,
                    "giga_product_category":x.product_category.value if x.product_category else "",
                    "giga_material":x.material.value if x.material else "",
                    "giga_work_types":json.dumps([w.value for w in x.work_types],ensure_ascii=False),
                    "giga_result_level":x.result_level,"giga_validation_status":x.validation_status,
                    "giga_reason":x.reason,"giga_evidence":json.dumps(x.evidence,ensure_ascii=False)})
            except GigaChatError as e: error=f"{type(e).__name__}: {e}"; error_kind=e.kind
            pd.DataFrame(results).to_csv(GIGA_CSV,index=False)
            pd.DataFrame(giga.cache.records()).to_csv(USAGE_CSV,index=False)
    return pd.DataFrame(results),pd.DataFrame(giga.cache.records())
if GIGACHAT_ENABLED and ALLOW_REAL_API_REQUESTS:
    giga_results,api_usage=await run_pilot(api_candidates)
else:
    giga_results=pd.read_csv(GIGA_CSV,keep_default_na=False) if GIGA_CSV.exists() else pd.DataFrame()
    api_usage=pd.read_csv(USAGE_CSV,keep_default_na=False) if USAGE_CSV.exists() else pd.DataFrame()
    print("API disabled; cached valid rows:",len(giga_results))
display(giga_results.head() if len(giga_results) else pd.DataFrame({"status":["no cached GigaChat results"]}))
selected_ids=set(api_candidates.product_id.astype(int))
cached_ids=set(giga_results.product_id.astype(int)) if len(giga_results) else set()
display(pd.DataFrame({"pilot_metric":["unique_products_selected","cached_classifications_for_selection","products_newly_sent_or_pending"],
 "count":[len(selected_ids),len(selected_ids&cached_ids),len(selected_ids-cached_ids)]}))
if len(api_usage) and "kind" in api_usage:
    request_breakdown=(api_usage.groupby(["kind","status","from_cache"],dropna=False).size()
                       .rename("attempts").reset_index())
    display(request_breakdown)


## 5. A. Overall, B. Paired, C. Hybrid, D. Errors and abstentions

Без human labels ниже измеряются agreement и coverage, не accuracy/precision.


In [ ]:
cmp=sample.copy()
if len(giga_results): cmp=cmp.merge(giga_results,on="product_id",how="left",validate="one_to_one")
else:
    for c in ["giga_decision","giga_product_category","giga_material","giga_work_types","giga_reason"]: cmp[c]=""
for c in ["giga_decision","giga_product_category","giga_material","giga_work_types"]: cmp[c]=cmp[c].fillna("")
cmp["ml_accept"]=cmp.eligible_0_6
cmp["giga_category_only"]=cmp.giga_decision.eq("accept")&cmp.giga_product_category.ne("")
cmp["giga_estimate_ready"]=cmp.giga_category_only&cmp.giga_material.ne("")&cmp.giga_work_types.ne("[]")&cmp.giga_work_types.ne("")
cmp["hybrid_category"]=cmp.ml_accept|(~cmp.ml_accept&cmp.giga_category_only)
cmp["hybrid_product_category"]=np.where(cmp.ml_accept,cmp.backend_product_category,np.where(cmp.giga_category_only,cmp.giga_product_category,""))
cmp["hybrid_estimate_ready"]=cmp.ml_accept&cmp.ml_work_types.ne("[]")
cmp.loc[~cmp.ml_accept,"hybrid_estimate_ready"]=cmp.loc[~cmp.ml_accept,"giga_estimate_ready"]

# A. Overall sample — denominator is always the preserved 240-product sample.
overall=pd.DataFrame({"metric":["ML category coverage","GigaChat eligible","GigaChat queried with valid result"],
 "count":[cmp.ml_accept.sum(),len(api_candidates),cmp.giga_decision.ne("").sum()],"denominator":len(cmp)})
overall["rate"]=overall["count"]/overall["denominator"]; display(overall)

# B. Paired comparison — exact products with a valid GigaChat decision.
paired=cmp[cmp.giga_decision.isin(["accept","reject","uncertain"])].copy()
paired_accept=paired[paired.giga_product_category.ne("")].copy()
paired_accept["category_agreement"]=paired_accept.giga_product_category.eq(paired_accept.backend_product_category)
display(pd.DataFrame({"paired_valid_results":[len(paired)],"paired_category_results":[len(paired_accept)],
                      "category_agreement":[paired_accept.category_agreement.mean() if len(paired_accept) else np.nan]}))
display(paired_accept.loc[~paired_accept.category_agreement,["product_id","name","backend_product_category",
                                                              "giga_product_category","giga_reason"]])

# C. Hybrid adds broad categories separately from estimate-ready bindings.
hybrid=pd.DataFrame({"approach":["ML","Hybrid"],"category_covered":[cmp.ml_accept.sum(),cmp.hybrid_category.sum()],
 "estimate_ready":[(cmp.ml_accept&cmp.ml_work_types.ne("[]")).sum(),cmp.hybrid_estimate_ready.sum()],"denominator":len(cmp)})
hybrid["category_coverage"]=hybrid.category_covered/hybrid.denominator
hybrid["estimate_ready_coverage"]=hybrid.estimate_ready/hybrid.denominator; display(hybrid)

# Threshold coverage uses the same denominator.
thresholds=pd.DataFrame({"threshold":[.6,.7,.8],
 "covered":[cmp.eligible_0_6.sum(),cmp.eligible_0_7.sum(),cmp.eligible_0_8.sum()]})
thresholds["denominator"]=len(cmp); thresholds["coverage"]=thresholds.covered/thresholds.denominator; display(thresholds)

# D. Valid abstentions are not technical errors.
valid_outcomes=cmp.giga_decision.replace({"":"not_queried"}).value_counts().rename_axis("outcome").to_frame("count")
errors=(api_usage.error_kind.replace({"":"none"}).value_counts().rename_axis("error_kind").to_frame("count")
        if len(api_usage) and "error_kind" in api_usage else pd.DataFrame())
display(valid_outcomes); display(errors)

# Category-level paired table, confidence distribution, and concrete examples.
display(pd.crosstab(paired_accept.backend_product_category,paired_accept.giga_product_category,margins=True))
cmp.groupby("sampling_stratum").ml_confidence.describe()[["count","mean","std","min","50%","max"]].pipe(display)
display(paired[["product_id","name","ml_confidence","backend_product_category","giga_decision",
                "giga_product_category","giga_reason"]].head(30))


## 6. Manual annotation template

Заполните отдельную копию `manual_annotations_completed.csv`. Human category — одна из 38 taxonomy values; work types — JSON list. Predictions и weak labels остаются отдельными колонками.


In [ ]:
cols=["product_id","name","catalog_category","sampling_stratum","ml_product_category","backend_material",
"ml_confidence","ml_work_types","giga_product_category","giga_work_types","hybrid_product_category",
"weak_product_category","weak_work_types","human_product_category","human_renovation_relevant",
"human_work_types","human_binding_acceptable","reviewer_comments"]
ann=cmp.copy()
for c in cols:
    if c not in ann: ann[c]=""
for c in ["human_product_category","human_renovation_relevant","human_work_types",
          "human_binding_acceptable","reviewer_comments"]: ann[c]=""
ann[cols].to_csv(TEMPLATE_CSV,index=False)
print(TEMPLATE_CSV); display(ann[cols].head(10))


## 7. Human-verified metrics

Accuracy/precision/recall вычисляются только при наличии независимого completed CSV. Coverage — доля non-abstain; invalid binding rate — доля предложений, отвергнутых reviewer. Без human labels показываются только agreement и coverage.


In [ ]:
def bools(s): return s.astype(str).str.lower().str.strip().map(
 {"true":True,"1":True,"yes":True,"да":True,"false":False,"0":False,"no":False,"нет":False})
if HUMAN_CSV.exists():
    human=pd.read_csv(HUMAN_CSV,keep_default_na=False)
    human=human[human.human_product_category.isin({x.value for x in ProductCategory})]
    ev=cmp.merge(human[["product_id","human_product_category","human_binding_acceptable",
      "human_renovation_relevant","human_work_types","reviewer_comments"]],on="product_id",validate="one_to_one")
    rows=[]
    for name,col in {"ML":"backend_product_category","GigaChat":"giga_product_category",
                     "Hybrid":"hybrid_product_category"}.items():
        covered=ev[col].ne(""); acceptable=bools(ev.human_binding_acceptable)
        rows.append({"approach":name,"human_rows":len(ev),"coverage":covered.mean(),
          "abstention_rate":1-covered.mean(),
          "accuracy_on_covered":(ev.loc[covered,col]==ev.loc[covered,"human_product_category"]).mean(),
          "invalid_binding_rate":(~acceptable[covered&acceptable.notna()]).mean()})
    display(pd.DataFrame(rows))
else:
    ev=pd.DataFrame(); print("Нет completed human annotations: accuracy/precision/recall не заявляются."); display(hybrid)


In [ ]:
if len(ev):
    labels=sorted(ev.human_product_category.unique()); fig,axes=plt.subplots(1,3,figsize=(18,5))
    for ax,(name,col) in zip(axes,{"ML":"backend_product_category","GigaChat":"giga_product_category",
                                  "Hybrid":"hybrid_product_category"}.items()):
        z=ev[ev[col].ne("")]
        ConfusionMatrixDisplay.from_predictions(z.human_product_category,z[col],labels=labels,
          include_values=False,xticks_rotation=90,colorbar=False,ax=ax); ax.set_title(name)
    plt.tight_layout(); plt.show()
    curve=[]
    for t in np.linspace(.4,.95,12):
        accepted=(ev.ml_confidence>=t)&~ev.taxonomy_conflict
        curve.append({"threshold":t,"coverage":accepted.mean(),
          "precision":(ev.loc[accepted,"backend_product_category"]==ev.loc[accepted,"human_product_category"]).mean()})
    pd.DataFrame(curve).plot(x="coverage",y="precision",marker="o",xlim=(0,1),ylim=(0,1),
                             title="ML precision vs coverage"); plt.show()
    fixed=ev[(ev.backend_product_category!=ev.human_product_category)&
             (ev.giga_product_category==ev.human_product_category)]
    harmed=ev[(ev.backend_product_category==ev.human_product_category)&ev.giga_product_category.ne("")&
              (ev.giga_product_category!=ev.human_product_category)]
    print("GigaChat corrected:",len(fixed),"introduced errors:",len(harmed))
    display(fixed.head(20)); display(harmed.head(20))


## 8. API usage и sequential projections

Используются только фактические usage/latency. Если измерений нет, projections не создаются. Cache hits не считаются новыми API затратами.


In [ ]:
display(api_usage if len(api_usage) else pd.DataFrame({"status":["no measured API usage"]}))
if len(api_usage):
    known=pd.to_numeric(api_usage.get("total_tokens"),errors="coerce")
    latency=pd.to_numeric(api_usage.get("elapsed_ms",api_usage.get("response_ms")),errors="coerce")
    display(pd.DataFrame({"reported_token_total":[known.sum(min_count=1)],"unknown_usage_rows":[known.isna().sum()],
                          "all_network_attempts":[(~api_usage.get("from_cache",False).astype(bool)).sum()]}))
    display(pd.DataFrame({"tokens":known.describe(),"latency_ms":latency.describe()}))
    print("Historical pilot: 30 responses, all reported usage = 34,109 tokens; 22 valid rows alone = 22,606.")


## 9. Опциональное объяснение уже выбранного товара

Контекст комнаты ниже синтетический и явно помечен. Товар реальный; используются только доступные fields/signals. Модель не может изменить выбор. Раздел классификации от explanations отделён.


In [ ]:
r=sample[sample.backend_material.eq("tile")].head(1)
if r.empty: r=sample.head(1)
x=r.iloc[0]
facts=[{"item_id":"synthetic:0","work":"tiling",
 "room":{"type":"bathroom","humidity":"high","fixture_is_synthetic":True},
 "product":{"id":int(x.product_id),"name":x["name"],"catalog_category":x.catalog_category},
 "selection_signals":{"ml_category":x.ml_product_category,"material":x.backend_material,
                      "ml_confidence":float(x.ml_confidence)},
 "budget_context":{"available":False}}]
display(pd.DataFrame(facts))
if RUN_OPTIONAL_EXPLANATION and GIGACHAT_ENABLED and ALLOW_REAL_API_REQUESTS:
    try: display(await giga.explain_estimate(None,facts))
    except GigaChatError as e: print("Unavailable:",e)
else: print("Real explanation disabled; selection unchanged.")


## Первый реальный пилот: 10–30 запросов

1. Добавьте credentials только в локальный `.env`.
2. Выполните connection test и выберите доступную модель.
3. Поставьте оба opt-in флага в `True`, сначала `MAX_API_PRODUCTS=10`.
4. Проверьте result/usage CSV и ручную таблицу. Только затем увеличьте до 30.
5. Cache и CSV обеспечивают resume. Не удаляйте cache, иначе запросы и токены повторятся.

**Интерпретация:** coverage — наличие решения; agreement — совпадение моделей; accuracy, precision и recall допустимы только с human ground truth. Рост coverage без precision не доказывает улучшение.
